# PatchDistill Colab Pilot

Run the clone/setup cells first. They copy the whole PatchDistill repository into the Colab runtime, then install dependencies from the repository root.

If `!pwd` is `/content` and `!ls` only shows `sample_data`, the repository is not on the Colab runtime yet. Opening this notebook from Cursor does not automatically copy `/home/blackleg/ws/research/llm/new/patchDistill` to Colab.

The next code cell uses the GitHub route:

```bash
git clone https://github.com/black-leg-nameko/patchDistill.git /content/patchDistill
cd /content/patchDistill
```

```python
# Route B: put the folder on Google Drive, then mount Drive
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/patchDistill
```

After that, the install cell below will find `requirements.txt` and `patchdistill/`.

In [1]:
!nvidia-smi

Sun Jun  7 18:39:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/black-leg-nameko/patchDistill.git"
REPO_DIR = Path("/content/patchDistill")

if REPO_DIR.exists():
    print(f"Repository already exists at {REPO_DIR}; pulling latest changes.")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("Using repo root:", Path.cwd())
print("Top-level files:")
print("\n".join(sorted(p.name for p in Path.cwd().iterdir())[:40]))

Using repo root: /content/patchDistill
Top-level files:
.git
.gitignore
README.md
configs
docs
notebooks
paper
patchdistill
pyproject.toml
requirements.txt
scripts
tests
研究計画書.md


In [3]:
MODEL_NAME = "gpt2"
RUN_NAME = "gpt2_a100_pilot"
LAYERS = "0,6,11"
N_DATA = 160
MAX_FEATURE_EXAMPLES = 80
MAX_PATCH_EXAMPLES = 12
MAX_PATCH_POSITIONS = 3
DTYPE = "bfloat16"
ATTN_IMPL = None

# For a stronger A100 pilot, try:
# MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# RUN_NAME = "qwen25_05b_a100_pilot"
# LAYERS = "0,12,23"
# ATTN_IMPL = "eager"

In [4]:
from pathlib import Path
import os
import subprocess
import sys

def find_repo_root():
    candidates = [Path.cwd(), Path('/content/patchDistill')]
    drive = Path('/content/drive/MyDrive')
    if drive.exists():
        candidates.extend([
            drive / 'patchDistill',
            drive / 'Colab Notebooks' / 'patchDistill',
        ])
    for candidate in candidates:
        if (candidate / 'requirements.txt').exists() and (candidate / 'patchdistill').is_dir():
            return candidate
    for root in [Path('/content'), drive]:
        if root.exists():
            for req in root.rglob('requirements.txt'):
                candidate = req.parent
                if (candidate / 'patchdistill').is_dir():
                    return candidate
    return None

repo_root = find_repo_root()
if repo_root is None:
    raise FileNotFoundError(
        'PatchDistill repo root was not found. Current Colab runtime does not contain the project files. '
        'If !pwd is /content and !ls only shows sample_data, clone/upload the whole repository first. '
        'Expected files: requirements.txt and patchdistill/.'
    )

os.chdir(repo_root)
print('Using repo root:', Path.cwd())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])

Using repo root: /content/patchDistill


0

In [5]:
!python -m patchdistill.cli make-data --n {N_DATA} --out data/synthetic_direct_pi.jsonl
!python -m patchdistill.cli run-surrogate --data data/synthetic_direct_pi.jsonl --out runs/surrogate_mvp --split template

{"out": "data/synthetic_direct_pi.jsonl", "n": 160}
{
  "out": "runs/surrogate_mvp",
  "models": {
    "tfidf_logreg": {
      "accuracy": 0.875,
      "precision": 0.8260869565217391,
      "recall": 0.95,
      "f1": 0.8837209302325582,
      "auroc": 0.9625,
      "auprc": 0.973562412342216,
      "false_negative_rate": 0.05,
      "false_positive_rate": 0.2
    },
    "rule_logreg": {
      "accuracy": 1.0,
      "precision": 1.0,
      "recall": 1.0,
      "f1": 1.0,
      "auroc": 1.0,
      "auprc": 1.0000000000000002,
      "false_negative_rate": 0.0,
      "false_positive_rate": 0.0
    },
    "patchdistill_surrogate": {
      "accuracy": 1.0,
      "precision": 1.0,
      "recall": 1.0,
      "f1": 1.0,
      "auroc": 1.0,
      "auprc": 1.0000000000000002,
      "false_negative_rate": 0.0,
      "false_positive_rate": 0.0
    }
  }
}


In [6]:
attn_arg = "" if ATTN_IMPL is None else f"--attn-implementation {ATTN_IMPL}"
!python -m patchdistill.cli hf-extract \
  --model {MODEL_NAME} \
  --data data/synthetic_direct_pi.jsonl \
  --out runs/{RUN_NAME}_features.jsonl \
  --max-examples {MAX_FEATURE_EXAMPLES} \
  --layers {LAYERS} \
  --dtype {DTYPE} \
  {attn_arg}

config.json: 100% 665/665 [00:00<00:00, 2.43MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 140kB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 5.60MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 3.22MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 36.4MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 548M/548M [00:02<00:00, 249MB/s]  
Loading weights: 100% 148/148 [00:00<00:00, 2483.76it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 582kB/s]
{"out": "runs/gpt2_a100_pilot_features.jsonl", "n": 80}


In [7]:
!python -m patchdistill.cli hf-patch \
  --model {MODEL_NAME} \
  --data data/synthetic_direct_pi.jsonl \
  --out runs/{RUN_NAME}_patch.jsonl \
  --layers {LAYERS} \
  --max-examples {MAX_PATCH_EXAMPLES} \
  --max-positions {MAX_PATCH_POSITIONS} \
  --dtype {DTYPE} \
  {attn_arg}

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 148/148 [00:00<00:00, 2689.17it/s]
{"out": "runs/gpt2_a100_pilot_patch.jsonl", "n": 12}


In [8]:
!python -m patchdistill.cli fit-proxy \
  --features runs/{RUN_NAME}_features.jsonl \
  --patch runs/{RUN_NAME}_patch.jsonl \
  --out runs/{RUN_NAME}_proxy

!python -m patchdistill.cli fit-detector \
  --features runs/{RUN_NAME}_features.jsonl \
  --out runs/{RUN_NAME}_detector_features_only

!python -m patchdistill.cli fit-detector \
  --features runs/{RUN_NAME}_features.jsonl \
  --patch runs/{RUN_NAME}_patch.jsonl \
  --out runs/{RUN_NAME}_detector_distilled

{
  "out": "runs/gpt2_a100_pilot_proxy",
  "mae": 0.06818077503505882,
  "mse": 0.016012894388297613
}
{
  "out": "runs/gpt2_a100_pilot_detector_features_only",
  "model": "hf_features_logreg",
  "metrics": {
    "accuracy": 1.0,
    "precision": 1.0,
    "recall": 1.0,
    "f1": 1.0,
    "auroc": 1.0,
    "auprc": 1.0000000000000002,
    "false_negative_rate": 0.0,
    "false_positive_rate": 0.0
  }
}
{
  "out": "runs/gpt2_a100_pilot_detector_distilled",
  "model": "hf_features_plus_distilled_patch_logreg",
  "metrics": {
    "accuracy": 1.0,
    "precision": 1.0,
    "recall": 1.0,
    "f1": 1.0,
    "auroc": 1.0,
    "auprc": 1.0000000000000002,
    "false_negative_rate": 0.0,
    "false_positive_rate": 0.0
  }
}


In [9]:
!python -m patchdistill.cli collect-results --runs runs --out runs/summary.json --markdown runs/summary.md

from pathlib import Path
print(Path("runs/summary.md").read_text())

{
  "out": "runs/summary.json",
  "markdown": "runs/summary.md",
  "n": 5
}
# PatchDistill Experiment Summary

## `gpt2_a100_pilot_detector_distilled/detector_metrics.json`
- model: `hf_features_plus_distilled_patch_logreg`
- F1: `1.0`
- AUROC: `1.0`
- FNR: `0.0`

## `gpt2_a100_pilot_detector_features_only/detector_metrics.json`
- model: `hf_features_logreg`
- F1: `1.0`
- AUROC: `1.0`
- FNR: `0.0`

## `gpt2_a100_pilot_proxy/proxy_metrics.json`
- MAE: `0.06818077503505882`
- MSE: `0.016012894388297613`
- n_aligned: `12`

## `surrogate_mvp/metrics.json`
- split: `template`
- n_total: `160`
- patchdistill_surrogate: F1=1.0, AUROC=1.0, FNR=0.0
- rule_logreg: F1=1.0, AUROC=1.0, FNR=0.0
- tfidf_logreg: F1=0.8837209302325582, AUROC=0.9625, FNR=0.05

## `surrogate_mvp/proxy_metrics.json`
- MAE: `0.05874425897624306`
- MSE: `0.009335313435483824`
- n_aligned: `None`



## Save Results To GitHub

Run this after an experiment if you want the `runs/` outputs archived under `artifacts/colab_runs/` and pushed back to GitHub. Use a fine-grained GitHub token with `Contents: Read and write` permission.

In [ ]:
import os
import getpass

if not os.environ.get("GITHUB_TOKEN"):
    os.environ["GITHUB_TOKEN"] = getpass.getpass("GitHub token: ")

archive_name = f"{RUN_NAME}_001"
!ARCHIVE_NAME={archive_name} scripts/save_colab_results_to_github.sh